# Residual CNN Practice

**Topic:** Modern CNN Architectures &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `X_train` `(4000, 16, 16)`, `y_train`: four texture classes (horizontal stripes, vertical stripes, checkerboard, diagonal) with random phase, spatial frequency, and strong pixel noise.
- `X_test` `(1200, 16, 16)`: predict these. `y_test` is used by the evaluation cell.

## Your task

1. Classify the four textures by any method. The intended route is a small CNN in PyTorch using at least one residual block (`out = relu(conv2(relu(conv1(x))) + x)`), trained with Adam.
2. Set `predictions` to your 0–3 test labels.

(The solution below shows the run-anywhere route, scikit-learn on FFT magnitudes, which needs no PyTorch.)

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

def textures(n, seed):
    r = np.random.default_rng(seed)
    y = r.integers(0, 4, n)
    X = np.zeros((n, 16, 16), dtype=np.float32)
    rows, cols = np.mgrid[0:16, 0:16]
    for i in range(n):
        freq = r.uniform(0.8, 1.6)
        phase = r.uniform(0, 2 * np.pi)
        if y[i] == 0:
            pat = np.sin(freq * rows + phase)
        elif y[i] == 1:
            pat = np.sin(freq * cols + phase)
        elif y[i] == 2:
            pat = np.sin(freq * rows + phase) * np.sin(freq * cols + phase)
        else:
            pat = np.sin(freq * (rows + cols) / np.sqrt(2) + phase)
        X[i] = pat + r.normal(0, 0.7, (16, 16))
    return X, y

X_train, y_train = textures(4000, 391)
X_test, y_test = textures(1200, 392)
print(f"X_train {X_train.shape}, 4 texture classes")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.8

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {int(_cls):>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

In Colab the intended build is a small ResNet-style CNN. The run-anywhere route below hands the classifier phase-robust frequency features, the magnitude of the 2-D FFT, which separates the four stripe/checker/diagonal textures cleanly.

```python
import numpy as np
from sklearn.neural_network import MLPClassifier

def fft_features(X):
    mag = np.abs(np.fft.rfft2(X))            # phase-robust magnitudes
    return mag.reshape(len(X), -1)

Xtr = fft_features(X_train)
Xte = fft_features(X_test)

mlp = MLPClassifier(hidden_layer_sizes=(128,), max_iter=500, random_state=0)
mlp.fit(Xtr, y_train)
predictions = mlp.predict(Xte)
```
